In [1]:
#%pip install transformers pillow
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
from transformers import AutoProcessor, AutoModel
from PIL import Image


In [2]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    load_dir = '/content/drive/MyDrive/xai-project5/results/feature_extraction'
    scripts_path = '/content/drive/MyDrive/xai-project5/src/scripts'
else:
    load_dir = os.path.abspath(os.path.join('..', 'results', 'feature_extraction'))
    scripts_path = os.path.abspath('../scripts')

if scripts_path not in sys.path:
    sys.path.append(scripts_path)

load_path = os.path.join(load_dir, 'nih_chest_embeddings.pt')
print(f"Caricamento configurato da: {load_path}")

Caricamento configurato da: /home/emmanuelmessina00/Scrivania/xai-project5/src/results/feature_extraction/nih_chest_embeddings.pt


Gli **Sparse Autoencoders (SAE)** implementano una forma di *sparse dictionary learning*, con l'obiettivo di apprendere una decomposizione sparsa di un segnale in un dizionario sovraccompleto di atomi.

---

Un SAE è costituito da:
- **Encoder** $W_{enc} \in \mathbb{R}^{d \times \omega}$: trasforma l'embedding di input in uno spazio latente
- **Decoder** $W_{dec} \in \mathbb{R}^{\omega \times d}$: ricostruisce l'embedding originale dallo spazio latente
- **Funzione di attivazione non-lineare** $\sigma : \mathbb{R}^{\omega} \to \mathbb{R}^{\omega}$
- **Bias condiviso** $b \in \mathbb{R}^d$: sottratto dall'input dell'encoder e aggiunto all'output del decoder

La larghezza dello strato latente $\omega$ è scelta come fattore della dimensione originale: $\omega := d \times \varepsilon$, dove $\varepsilon$ è il **fattore di espansione**.

---

Dato un embedding $v \in \mathbb{R}^d$, il SAE decompose il vettore in:
- **Vettore di attivazioni**: $\phi(v) := \sigma(W_{enc}^{\top}(v - b))$
- **Vettore ricostruito**: $\hat{v} := W_{dec}^{\top}\phi(v) + b$

---

La loss function combina un **obiettivo di ricostruzione** con una **regolarizzazione di sparsità**:

$$\mathcal{L}(v) = R(v) + \lambda S(v)$$

dove:
- **Ricostruzione (L2)**: $R(v) := \|v - \hat{v}\|_2^2$ garantisce la fedeltà dell'informazione
- **Sparsità (L1)**: $S(v) := \|\phi(v)\|_1$ penalizza l'attivazione di troppi neuroni latenti
- **Hyperparameter** $\lambda$: regola il trade-off tra ricostruzione e sparsità


---

L'implementazione completa del SAE è disponibile nel file `scripts/sae.py`, dove sono definiti:
- La classe `SparseAutoencoder` con i metodi `forward()` e `normalize_decoder_weights()`
- La funzione di loss `sae_loss_function()` che calcola il trade-off tra ricostruzione e sparsità

## Estrazione degli Embedding con CLIP

In questa sezione, utilizziamo il modello **PubMed CLIP** ([flaviagiammarino/pubmed-clip-vit-base-patch32](https://huggingface.co/flaviagiammarino/pubmed-clip-vit-base-patch32)) per estrarre gli embedding visivi dalle immagini mediche.

PubMed CLIP è una variante specializzata del modello CLIP (Contrastive Language-Image Pre-training) addestrata specificamente su dati biomedici. Questo modello fornisce:

- **Vision Encoder (ViT-Base)**: Trasforma le immagini mediche in embedding vettoriali di dimensione 512
- **Text Encoder**: Trasforma descrizioni testuali in embedding dello stesso spazio latente
- **Allineamento multimodale**: Gli embedding visivi e testuali sono proiettati nello stesso spazio, permettendo il calcolo della similarità coseno

Gli embedding estratti sono **normalizzati L2**, garantendo che ogni vettore abbia norma unitaria. Questo è fondamentale per:
1. Stabilizzare il training dello Sparse Autoencoder
2. Semplificare il calcolo della similarità coseno (che diventa una semplice moltiplicazione matriciale)
3. Interpretare semanticamente i neuroni latenti del SAE

In [3]:
model_id = "flaviagiammarino/pubmed-clip-vit-base-patch32"

print(f"Scaricamento del modello {model_id} in corso...")
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

Scaricamento del modello flaviagiammarino/pubmed-clip-vit-base-patch32 in corso...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

### Training of Sparse AutoEncoder

In [ ]:
#%load_ext autoreload
#%autoreload 2
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sae import SparseAutoencoder, sae_loss_function

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizzando il device: {device}")

print("Carichiamo il dataset presente in nih_chest_embeddings.pt...")
vision_embeddings=torch.load(load_path).to(device)
dataset= TensorDataset(vision_embeddings)

batch_size=256
dataloader=DataLoader(dataset,batch_size=batch_size,shuffle=True)

sae = SparseAutoencoder(input_dim=512, hidden_dim=2048).to(device)
learning_rate = 1e-3
optimizer = optim.Adam(sae.parameters(), lr=learning_rate)

l1_lambda = 1e-4
num_epochs = 20
print("ADDESTRAMENTO SAE...")
for epoch in range(num_epochs):
    sae.train()
    epoch_total_loss=0.0
    epoch_mse_loss=0.0
    epoch_l1_loss=0.0

    for batch in dataloader:
        x=batch[0].to(device)
        #forward 
        x_hat,z=sae(x)

        #calcolo loss
        total_loss, mse_loss, l1_loss = sae_loss_function(x, x_hat, z, l1_lambda)
        #backward e ottimizzazione
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        sae.normalize_decoder_weights() #si forza ad 1 per evitare il collasso

        #aggiorniamo le metriche
        epoch_total_loss += total_loss.item()
        epoch_mse_loss += mse_loss.item()
        epoch_l1_loss += l1_loss.item()
    
    avg_total_loss = epoch_total_loss / len(dataloader)
    avg_mse = epoch_mse_loss / len(dataloader)
    avg_l1 = epoch_l1_loss / len(dataloader)

    if (epoch + 1) % 5 == 0:
        print(f"Epoca [{epoch+1:02d}/{num_epochs}] | "
              f"Loss Tot: {avg_total_loss:.4f} | "
              f"MSE (Ricostruzione): {avg_mse:.4f} | "
              f"L1 (Sparsità): {avg_l1:.4f}")

Utilizzando il device: cpu
Carichiamo il dataset presente in nih_chest_embeddings.pt...
ADDESTRAMENTO SAE...
Epoca [05/20] | Loss Tot: 0.0000 | MSE (Ricostruzione): 0.0000 | L1 (Sparsità): 0.0051
Epoca [10/20] | Loss Tot: 0.0000 | MSE (Ricostruzione): 0.0000 | L1 (Sparsità): 0.0052
Epoca [15/20] | Loss Tot: 0.0000 | MSE (Ricostruzione): 0.0000 | L1 (Sparsità): 0.0052
Epoca [20/20] | Loss Tot: 0.0000 | MSE (Ricostruzione): 0.0000 | L1 (Sparsità): 0.0051




L'addestramento non supervisionato del nostro *Sparse Autoencoder* (SAE) ha prodotto un dizionario sovracompleto di feature: la matrice dei pesi del decoder, $W_{dec} \in \mathbb{R}^{512 \times 2048}$. Ogni colonna di questa matrice rappresenta un singolo "concetto visivo" che il modello ha isolato autonomamente guardando i pixel. Tuttavia, questi concetti sono matematicamente "muti": il modello ne riconosce l'esistenza, ma non possiede un'etichetta semantica umana per descriverli.

L'assegnazione delle etichette (Grounding) avviene confrontando i concetti del SAE con gli embedding testuali. Poiché entrambi i tensori sono stati preventivamente assoggettati a **normalizzazione L2** (rendendo i vettori di norma unitaria), il calcolo della similarità coseno si riduce a una singola moltiplicazione matriciale:

$$S = T \cdot W_{dec}$$

Dove $T$ è la matrice degli embedding testuali e $W_{dec}$ è il dizionario del SAE. Estraendo i valori massimi (*Top-K*) da questo prodotto, identifichiamo in modo inequivocabile quali specifici neuroni dell'Autoencoder si sono specializzati nel rilevare determinati concetti medici, rendendo la rappresentazione finale totalmente interpretabile.

### Spiegabilità Globale: Costruzione del Dizionario Semantico

La spiegabilità globale si pone l'obiettivo di mappare e comprendere l'architettura interna del modello nella sua interezza, indipendentemente dai singoli campioni di input. In questo paradigma, l'intento è costruire un vero e proprio dizionario che colleghi i concetti visivi latenti, appresi in modo totalmente non supervisionato, al linguaggio umano. Dal punto di vista procedurale, questo processo, noto come *semantic grounding*, si realizza confrontando gli embedding testuali dei concetti clinici noti con i pesi del dizionario appreso dallo Sparse Autoencoder. 

Poiché entrambi i vettori vengono assoggettati a una preventiva normalizzazione L2, il calcolo della similarità coseno si riduce elegantemente a una singola moltiplicazione matriciale definita come $S = T \cdot W_{dec}$, dove $T$ rappresenta la matrice degli embedding testuali e $W_{dec}$ costituisce il dizionario del decoder del SAE. Il risultato di questa operazione geometrica permette di identificare a priori l'allineamento a riposo della rete, stabilendo per ogni parola clinica fornita in input lo specifico neurone latente che ne codifica la rappresentazione visiva all'interno dello spazio ad alta dimensionalità.

In [ ]:
#i nostri concetti (TODO! Implementare l'architettura UMLS + LLM per il dizionario )
medical_concepts = [
    "healthy lungs", 
    "bone fracture", 
    "pneumonia", 
    "pleural effusion",
    "heart",
    "ribs",
    "medical imaging artifact"
]

text_inputs=processor(text=medical_concepts,padding=True,return_tensors='pt') #we pass to encoder the textual concepts

with torch.no_grad():
    text_outputs=model.get_text_features(**text_inputs) 
    text_tensor = text_outputs.pooler_output
#here we obtain the corresponding embeddings 
text_embeddings=F.normalize(text_tensor, p=2,dim=1).to(device)

with torch.no_grad():
    sae_dictionary=F.normalize(sae.decoder.weight.data,p=2,dim=0)  

similarities=torch.matmul(text_embeddings,sae_dictionary) #calcoliamo la similarità 

top_k = 3
for idx, concept in enumerate(medical_concepts):
    concept_sims = similarities[idx]
    # Otteniamo i primi k valori e i loro indici (i "neuroni" del SAE)
    top_values, top_indices = torch.topk(concept_sims, top_k)
    
    print(f"\nConcetto testuale: '{concept}'")
    for i in range(top_k):
        print(f"  -> Neurone SAE {top_indices[i].item():4d} (Similarità: {top_values[i].item():.4f})")


Concetto testuale: 'healthy lungs'
  -> Neurone SAE  955 (Similarità: 0.1479)
  -> Neurone SAE 1880 (Similarità: 0.1334)
  -> Neurone SAE  224 (Similarità: 0.1326)

Concetto testuale: 'bone fracture'
  -> Neurone SAE  992 (Similarità: 0.1609)
  -> Neurone SAE 1029 (Similarità: 0.1311)
  -> Neurone SAE  800 (Similarità: 0.1187)

Concetto testuale: 'pneumonia'
  -> Neurone SAE 1880 (Similarità: 0.1481)
  -> Neurone SAE 1134 (Similarità: 0.1441)
  -> Neurone SAE 1921 (Similarità: 0.1394)

Concetto testuale: 'pleural effusion'
  -> Neurone SAE 1880 (Similarità: 0.1710)
  -> Neurone SAE  955 (Similarità: 0.1430)
  -> Neurone SAE 1029 (Similarità: 0.1419)

Concetto testuale: 'heart'
  -> Neurone SAE 1880 (Similarità: 0.1500)
  -> Neurone SAE  215 (Similarità: 0.1276)
  -> Neurone SAE 1921 (Similarità: 0.1226)

Concetto testuale: 'ribs'
  -> Neurone SAE 1880 (Similarità: 0.1505)
  -> Neurone SAE  567 (Similarità: 0.1358)
  -> Neurone SAE 1105 (Similarità: 0.1339)

Concetto testuale: 'medical

### Spiegabilità Locale: Interpretazione del Singolo Campione

A differenza dell'approccio globale, la spiegabilità locale si concentra sull'interpretazione dinamica del comportamento del modello rispetto a un singolo e specifico dato di input. L'obiettivo in questa fase è interrogare la rete neurale per ottenere una spiegazione che illustri quali concetti visivi sono stati rilevati all'interno di una particolare radiografia nel momento esatto in cui viene processata. 

Fornendo l'immagine in input al Vision-Language Model e proiettandone il vettore risultante all'interno dell'encoder dello Sparse Autoencoder, si ottiene un vettore di attivazioni latenti $z$. In questo frangente, la rete reagisce allo stimolo visivo accendendo fisicamente una frazione molto ristretta dei suoi neuroni. Analizzando l'intensità di queste attivazioni e sfruttando la matrice di similarità $S$ precedentemente calcolata nella fase di spiegabilità globale, è possibile mappare i neuroni maggiormente stimolati sul relativo concetto testuale. Questo meccanismo traduce le attivazioni matematiche in una diagnosi interpretativa leggibile in linguaggio naturale, evidenziando esattamente cosa il modello sta osservando nella singola istanza clinica.

In [14]:
def explain_single_image(path, vlm_model, vlm_processor, sae_model, concept_similarities, concept_names, device, top_k_neurons=5):

    vlm_model.eval()
    sae_model.eval()
    
    image = Image.open(path)

    if image.mode != "RGB":
        image = image.convert("RGB")

    inputs = vlm_processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        
        vision_outputs = vlm_model.get_image_features(**inputs)
        # Extracting the tensor
        vision_tensor = vision_outputs.pooler_output
        vision_embeddings = F.normalize(vision_tensor, p=2, dim=1)

        _, z = sae_model(vision_embeddings)

    activations=z[0] #Extracting the activation vectors
    top_activation_values, top_neuron_indices = torch.topk(activations, top_k_neurons)

    found_concepts = False
    for val, neuron_idx in zip(top_activation_values, top_neuron_indices):
        if val.item() <= 0.01: 
            continue 
            
        found_concepts = True
        print(f"\n[Neurone {neuron_idx.item():4d}] -> Attivazione: {val.item():.4f}")
        neuron_concept_scores = concept_similarities[:, neuron_idx]
        
        best_concept_idx = torch.argmax(neuron_concept_scores).item()
        best_concept_score = neuron_concept_scores[best_concept_idx].item()
        
        print(f"  Concept name: '{concept_names[best_concept_idx]}'")
        print(f"  Concept score: {best_concept_score:.4f}")

    if not found_concepts:
        print("\n No sufficient strong activations.")


explain_single_image(
    path="../images/raggi_x.jpg", 
    vlm_model=model, 
    vlm_processor=processor, 
    sae_model=sae, 
    concept_similarities=similarities, 
    concept_names=medical_concepts, 
    device=device
)


[Neurone 1894] -> Attivazione: 0.1268
  Concept name: 'heart'
  Concept score: 0.1148

[Neurone  812] -> Attivazione: 0.1092
  Concept name: 'bone fracture'
  Concept score: 0.0267

[Neurone 1587] -> Attivazione: 0.1037
  Concept name: 'bone fracture'
  Concept score: -0.0160

[Neurone 1669] -> Attivazione: 0.0976
  Concept name: 'bone fracture'
  Concept score: 0.0751

[Neurone  268] -> Attivazione: 0.0966
  Concept name: 'medical imaging artifact'
  Concept score: 0.0585
